# 01 — Halo Catalogue

**Purpose:** Build the filtered MDPL2 halo catalogue used for cluster masking.

This notebook loads the raw MDPL2 lightcone slice files (`haloslc_rot_*.npz`), filters
to clusters with M500c ≥ 3×10¹⁴ M☉ (the paper's cluster masking threshold), concatenates
all slices into a single catalogue, and saves it as a `.npy` file for use in
`02_masking.ipynb`.

**Inputs:**
- Raw halo lightcone slices: `haloslc/haloslc_rot_*.npz` (on remote cluster)

**Outputs:**
- Filtered halo catalogue: `data/halo_catalogue/halo_catalogue_m500gt3e14.npz`

**Key module functions:** none — this notebook only uses `numpy` and standard I/O.

**Paper reference:** §2 (cluster masking, M500c ≥ 3×10¹⁴ M☉ threshold).

In [5]:
import glob
import numpy as np
from pathlib import Path

HALO_DIR = "~/rds/hpc-work/haloslc"
M500C_THRESHOLD = 3e14  # M_sun — paper's cluster masking threshold
OUT_PATH = "~/rds/hpc-work/halo_catalogue/halo_catalogue_m500gt3e14"


In [6]:
# Discover all lightcone slice files
slice_files = sorted(glob.glob(f"{HALO_DIR}/haloslc_rot_*.npz"))
print(f"Found {len(slice_files)} lightcone slice files")

# Inspect the first file to understand its structure
sample_data = np.load(slice_files[0], allow_pickle=True)
print(f"Keys: {list(sample_data.keys())}")
first_arr = sample_data[list(sample_data.keys())[0]]
print(f"Shape: {first_arr.shape}  dtype: {first_arr.dtype}")
print("Columns assumed: ra, dec, z, m200c, m500c, vlos, vtht, vphi")


Found 0 lightcone slice files


IndexError: list index out of range

In [7]:
# Load every slice, filter by M500c >= threshold, and concatenate
all_halos = []
for fpath in slice_files:
    data = np.load(fpath, allow_pickle=True)
    arr = data[list(data.keys())[0]]  # shape (N_halos, 8)
    # column layout: ra, dec, z, m200c, m500c, vlos, vtht, vphi
    m500c_col = arr[:, 4]
    mask = m500c_col >= M500C_THRESHOLD
    all_halos.append(arr[mask])
    
catalogue = np.concatenate(all_halos, axis=0)
print(f"Total halos with M500c >= {M500C_THRESHOLD:.0e} M_sun: {len(catalogue):,}")
print(f"Catalogue shape: {catalogue.shape}")
print(f"Redshift range: {catalogue[:, 2].min():.3f} – {catalogue[:, 2].max():.3f}")
print(f"M500c range:    {catalogue[:, 4].min():.2e} – {catalogue[:, 4].max():.2e} M_sun")


ValueError: need at least one array to concatenate

In [ ]:
Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
np.save(OUT_PATH, catalogue)
print(f"Saved catalogue → {OUT_PATH}")
